# cordslite API

In [ ]:
#| default_exp core

In [ ]:
#| export
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from fastcore.meta import *
from fastcore.utils import *
from fasthtml.common import *
from nacl.bindings import crypto_aead_xchacha20poly1305_ietf_decrypt as xchacha_decrypt
from nacl.bindings import crypto_aead_xchacha20poly1305_ietf_encrypt as xchacha_encrypt

import asyncio,davey,ffmpeg,httpx,json,os,random,re,socket,struct,time
import numpy as np
import websockets, websockets.asyncio.client
try: import opuslib_next
except: print("Failed to import opuslib-next")

In [ ]:
from IPython.display import Audio

import wave

Discord has three main APIs:
- **REST API** - for actions (send message, get channels, fetch history). Request/response style.
- **Gateway API** - for real-time events via WebSocket (new messages, reactions, user joins).
- **Voice API** - for real-time streaming of audio via UDP

We start with the REST API. To use it, you need a **bot token** from the [Discord Developer Portal](https://discord.com/developers/applications):
1. Create an application
2. Go to "Bot" section and create a bot
3. Copy the token (keep it secret!)
4. Under "OAuth2 → URL Generator", select `bot` scope and choose permissions (e.g., `Send Messages`, `Read Message History`)
5. Use the generated URL to invite the bot to your server

The `DiscordClient` wraps `httpx.Client` with Discord's base URL (`https://discord.com/api/v10`) and auth headers pre-configured.

In [ ]:
#| export
class DiscordClient:
    def __init__(self, token=None, user_token=None, name='cordslite', ver='0.1'):
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.user_token = user_token or os.environ.get('DISCORD_USER_TOKEN')
        self.base_url = 'https://discord.com/api/v10'
        self.headers = {'Authorization': f'Bot {self.token}', 'User-Agent': f'DiscordBot ({name}, {ver})'}
        self.cli = httpx.AsyncClient(base_url=self.base_url, headers=self.headers)


The token defaults to the `DISCORD_BOT_TOKEN` environment variable—no hardcoding secrets!

In [ ]:
dc = DiscordClient()


## REST endpoints

Every REST call goes through `_req`, which handles two concerns automatically:
- **Rate limiting** — if Discord returns `429 Too Many Requests`, we wait the `Retry-After` duration and retry.
- **Error handling** — non-2xx responses are raised as `DiscordError` with Discord's error code, message, and HTTP status, so callers don't need to check responses manually.

In [ ]:
#| export
class DiscordError(Exception):
    def __init__(self, code, msg, status): self.code,self.msg,self.status = code,msg,status
    def __str__(self): return f"DiscordError({self.status}): [{self.code}] {self.msg}"

@patch
async def _req(self:DiscordClient, method, path, data=None, files=None, use_user=False, **kw):
    kw = filter_values(kw, negate(is_(None)))
    if method=='GET': params,json = kw,None
    else: params,json = None,kw
    if use_user:
        if not self.user_token: raise ValueError("User token required for this operation")
        kw['headers'] = {'Authorization': self.user_token}
    r = await self.cli.request(method, path, data=data, files=files, params=params, json=json)
    if r.status_code == 429:
        await asyncio.sleep(float(r.headers.get('Retry-After', 1))+0.1)
        return await self._req(method, path, use_user=use_user, **kw)
    if r.status_code >= 400:
        try: d = r.json()
        except Exception: d = {}
        raise DiscordError(d.get('code', 0), d.get('message', r.text[:200]), r.status_code)
    if r.status_code == 204: return 'done'
    try: return r.json()
    except Exception as e: raise Exception(f"Failed to parse response: {r.text}; {e}")

Discord's REST endpoints return JSON with many fields. Rather than defining properties for every field, we use a flexible base class pattern:

`DiscordObject` provides `__getitem__` and `__getattr__` so you can access data as `obj.name` or `obj['name']`. The `__dir__` method enables autocomplete in notebooks and solveit dialogs—just type `obj.` and see all available fields!

This means when Discord adds new fields to their API, our code automatically supports them without changes.

In [ ]:
#| export
class DiscordObject(AttrDict):
    def __init__(self, data, client):
        super().__init__(data)
        self._client = getattr(client, '_client', client)
        self._parent = client
    
    async def __call__(self, meth, path, **kw): return await self._client._req(meth, path, **kw)
    def obj(self, nm, d): return globals()[nm](d, self)
    def coll(self, nm, ds): return globals()[nm+'s'](self.obj(nm,d) for d in ds)

Discord's hierarchy is **Guild → Channels → Messages**. A Guild (server) contains channels, and channels contain messages. Each has its own REST endpoints:
- `GET /guilds/{id}` - fetch guild info
- `GET /guilds/{id}/channels` - list channels
- `GET /channels/{id}/messages` - fetch messages
- `POST /channels/{id}/messages` - send a message

We build wrapper classes for each, inheriting from `DiscordObject` and just adding a nice `__repr__`.

In [ ]:
#| export
class Guild(DiscordObject):
    def __repr__(self): return f"Guild(id={self.id}, name={self.name!r})"

We use `@patch` from fastcore to add methods incrementally—great for interactive development where you want to test each piece as you build it. This `guild` method fetches guild data and wraps it in our `Guild` class.

In [ ]:
#| export
@patch
async def guild(self:DiscordClient, guild_id):
    return Guild(await self._req('GET', f'/guilds/{guild_id}'), self)

In [ ]:
gid = '1493461895615873044'
# gid = '1327046393453613076'
gld = await dc.guild(gid); gld

<div class="prose" markdown="1">

```python
Guild(id=1493461895615873044, name='AAI-test-dev')
```

</div>

Channels have a `type` field: 0=text, 2=voice, 4=category. The `Channels` class inherits from `list` and adds `_repr_html_` for nice table display in notebooks/solveit. This pattern—a wrapper class plus a collection class with HTML repr—makes exploring the API much more pleasant.

In [ ]:
#| export
def html_table(items, hdrs, fn):
    if not items: return ""
    rows = (Tr(map(Td, fn(o))) for o in items)
    return to_xml(Table(Thead(Tr(map(Th, hdrs))), Tbody(rows), cls="prose"))

In [ ]:
#| export
class Channel(DiscordObject):
    @property
    async def guild(self):
        if not hasattr(self, '_gld'): self._gld = await self._client.guild(self.guild_id)
        return self._gld
    def __repr__(self): return f"Channel(id={self.id}, name={self.name!r}, type={self.type})"

class Channels(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Type"), lambda c: (c.id, c.name, c.type))

In [ ]:
#| export
@patch
async def channels(self:Guild, limit=None):
    data = await self('GET', f'/guilds/{self.id}/channels')
    if limit: data = data[:limit]
    return self.coll('Channel', data)

In [ ]:
chs = await gld.channels()
chs

ID,Name,Type
1493461896139903026,Text Channels,4
1493461896139903027,Voice Channels,4
1493461896139903028,general,0
1493461896139903029,General,2


In [ ]:
#| export
@patch
async def channel(self:DiscordClient, channel_id):
    return Channel(await self._req('GET', f'/channels/{channel_id}'), self)

In [ ]:
chid = '1493461896139903028'
# chid = '1327046393453613079'

In [ ]:
ch = await dc.channel(chid)
ch

<div class="prose" markdown="1">

```python
Channel(id=1493461896139903028, name='general', type=0)
```

</div>

Messages are the core of most bot functionality. Note that Discord returns messages in reverse chronological order (newest first), so we `reverse()` the data to get chronological order. The table shows a preview of content, author, and timestamp.

Note: To read message content, your bot needs the `MESSAGE_CONTENT` privileged intent enabled in the Developer Portal! Without it, the `content` field will be empty for messages not sent by your bot or mentioning it.

In [ ]:
#| export
class Message(DiscordObject):
    def __init__(self, data, dobj):
        super().__init__(data, dobj)
        self.raw_content = self.content
        mentions = {m['id']: m['username'] for m in data.get('mentions', [])}
        def repl(m): return f"@{mentions.get(m.group(1), 'unknown')}"
        self['content'] = re.sub(r'<@!?(\d+)>', repl, self.content)

    def __repr__(self): 
        author = self.author['username']
        preview = self.content[:30] + '…' if len(self.content) > 30 else self.content
        return f"Message(id={self.id}, author={author!r}, content={preview!r})"

    @property
    async def channel(self):
        if not hasattr(self, '_ch'): self._ch = Channel(await self('GET', f'/channels/{self.channel_id}'), self._client)
        return self._ch

In [ ]:
#| export
class Messages(list):
    def _repr_html_(self):
        return html_table(self, ("ID", "Author", "Content", "Date"),
            lambda m: (m.id, m.author['username'], m.content[:50], m.timestamp[:10]))

In [ ]:
#| export
@patch
async def messages(self:Channel, limit=50, before=None, after=None, around=None):
    "Fetch channel messages. `before`, `after`, and `around` are mutually exclusive message IDs."
    if sum(x is not None for x in [before, after, around]) > 1:
        raise ValueError("Pass only one of `before`, `after`, or `around`")

    def mid(x): return x.id if isinstance(x, Message) else x
    data = await self('GET', f'/channels/{self.id}/messages',
                      limit=limit, before=mid(before), after=mid(after), around=mid(around))
    return self.coll('Message', reversed(data))

In [ ]:
msgs = await ch.messages(5)
msgs

ID,Author,Content,Date
1493727509747990650,jeremyhoward,yo,2026-04-14
1493727514118324275,jeremyhoward,yo,2026-04-14
1493727591461290116,share2self,"Hi, from Solveit!",2026-04-14
1493727595722833950,share2self,I'm replying to myself 🤓,2026-04-14
1493727601154461717,share2self,Here is a file!,2026-04-14


In [ ]:
#| export
@patch
async def before(self:Message, limit=5):
    "Fetch messages before this message in the same channel/thread."
    return await (await self.channel).messages(before=self, limit=limit)

@patch
async def after(self:Message, limit=5):
    "Fetch messages after this message in the same channel/thread."
    return await (await self.channel).messages(after=self, limit=limit)

@patch
async def around(self:Message, limit=11):
    "Fetch messages around this message in the same channel/thread."
    return await (await self.channel).messages(around=self, limit=limit)

In [ ]:
#| export
DISCORD_WEB = 'https://discord.com/channels'

@patch(as_prop=True)
def url(self:Guild): return f'{DISCORD_WEB}/{self.id}'

@patch(as_prop=True)
def url(self:Channel): return f'{DISCORD_WEB}/{self.guild_id}/{self.id}'

@patch(as_prop=True)
def url(self:Message):
    gid = self.get('guild_id') or self._parent.get('guild_id') or self._parent.id
    return f'{DISCORD_WEB}/{gid}/{self.channel_id}/{self.id}'

In [ ]:
gld.url, ch.url, msgs[0].url

('https://discord.com/channels/1493461895615873044',
 'https://discord.com/channels/1493461895615873044/1493461896139903028',
 'https://discord.com/channels/1493461895615873044/1493461896139903028/1493727509747990650')

Guild search uses snowflake IDs for date filtering (`min_id`/`max_id`), but we autoconvert `'YYYY-MM-DD'` strings for convenience for `before/after`.

In [ ]:
#| export
depoch = 1420070400000

def date2snowflake(date_str):
    "Convert 'YYYY-MM-DD' to a Discord snowflake ID"
    dt = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)
    return str(int((dt.timestamp() * 1000 - depoch) * (2**22)))

@patch
async def search(self:Guild, content=None, author_id=None, channel_id=None, mentions=None,
                 has=None, before=None, after=None, pinned=None, sort_by=None, sort_order=None,
                   offset=None, limit=None, use_user=False, nothread:bool=True):
    "Search guild messages. `before`/`after` accept 'YYYY-MM-DD' strings or snowflake IDs."
    if before and not str(before).isdigit(): before = date2snowflake(before)
    if after and not str(after).isdigit(): after = date2snowflake(after)
    r = await self('GET', f'/guilds/{self.id}/messages/search', use_user=use_user,
        content=content, author_id=author_id, channel_id=channel_id,
        mentions=mentions, has=has, min_id=after, max_id=before, pinned=pinned,
        sort_by=sort_by, sort_order=sort_order, offset=offset, limit=limit)
    msgs = [m for m in [m[0] for m in r['messages']]
        if not (nothread and channel_id and m['channel_id'] != channel_id)]
    return self.coll('Message', msgs)

In [ ]:
msgs = await gld.search(after='2026-02-16', limit=5)
msgs

ID,Author,Content,Date
1493727601154461717,share2self,Here is a file!,2026-04-14
1493727595722833950,share2self,I'm replying to myself 🤓,2026-04-14
1493727591461290116,share2self,"Hi, from Solveit!",2026-04-14
1493727514118324275,jeremyhoward,yo,2026-04-14
1493727509747990650,jeremyhoward,yo,2026-04-14


Sometimes you need to search by name rather than snowflake ID. `find_member` searches the guild's members by username, nickname, or display name using Discord's member search endpoint, and returns the first match's user ID. This makes it easy to chain into `search`.

In [ ]:
#| export
@patch
async def find_member(self:Guild, name):
    "Search guild members by name/nick, return first match's user ID or None"
    data = await self( 'GET', f'/guilds/{self.id}/members/search', query=name, limit=1)
    return data[0]['user']['id'] if data else None

In [ ]:
uid = await gld.find_member('nate.dawgg')
assert uid
await gld.search(author_id=uid, limit=5)

ID,Author,Content,Date
1493722755617656893,nate.dawgg,AAI vc?,2026-04-14
1493722195598250034,nate.dawgg,yes please <:nerdcowboy:1146305709416390740>,2026-04-14
1493721454661861386,nate.dawgg,oh yes please!,2026-04-14
1493712817293889648,nate.dawgg,haha I see someone is running the core nb! Please,2026-04-14
1493531161941770353,nate.dawgg,,2026-04-14


Sending messages is a POST request with JSON body. Pass `reply_id` to thread a reply under an existing message. For file attachments, we switch to `multipart/form-data`—Discord expects a `payload_json` field with the message JSON, plus `files[n]` fields for each file.

In [ ]:
#| export
@patch
async def send(self:Channel, content='', files=None, reply_id=None):
    "Send a message with optional file attachments"
    path = f'/channels/{self.id}/messages'
    mref = {'message_id': reply_id} if reply_id else None
    payload = filter_values(dict(content=content, message_reference=mref), negate(is_(None)))
    fs = [('files[%d]' % i, (f.name, f.read_bytes(), None)) for i,f in enumerate(map(Path, listify(files)))]
    r = await self('POST', path, data={'payload_json': json.dumps(payload)}, files=fs)
    return Message(r, self)

In [ ]:
msg = await ch.send('Hi, from Solveit!'); msg

<div class="prose" markdown="1">

```python
Message(id=1522409143309172766, author='share2self', content='Hi, from Solveit!')
```

</div>

In [ ]:
reply_msg = await ch.send("I'm replying to myself 🤓", reply_id=msg.id); reply_msg

<div class="prose" markdown="1">

```python
Message(id=1522409149483319380, author='share2self', content="I'm replying to myself 🤓")
```

</div>

In [ ]:
await msg.channel

<div class="prose" markdown="1">

```python
Channel(id=1493461896139903028, name='general', type=0)
```

</div>

In [ ]:
msg = await ch.send('Here is a file!', files=['../README.md']); msg

<div class="prose" markdown="1">

```python
Message(id=1522409165291782254, author='share2self', content='Here is a file!')
```

</div>

In [ ]:
#| export
@patch
@delegates(Guild.search, but=['channel_id'])
async def search(self:Channel, **kwargs): return await (await self.guild).search(channel_id=self.id, **kwargs)

In [ ]:
await ch.search(has='file', limit=1)

ID,Author,Content,Date
1522409165291782254,share2self,Here is a file!,2026-07-03


Discord messages can have file attachments. The `Attachment` class wraps them as `DiscordObject`s—giving you attribute access to `filename`, `size`, `content_type`, `url`, etc. The `fetch` method downloads the file content using the existing httpx client.

In [ ]:
#| export
class Attachment(DiscordObject):
    async def fetch(self): return (await self._client.cli.get(self.url)).content
    def __repr__(self): return f"Attachment(filename={self.filename!r}, size={self.size}, type={self.content_type})"
    def _repr_html_(self):
        if self.content_type.startswith('image/'):
            return f'<img src="{self.url}" style="max-width:400px"><br><small>{self.filename} ({self.size:,}B)</small>'
        return f'<code>{self.filename}</code> ({self.content_type}, {self.size:,}B)'

@patch(as_prop=True)
def attachments(self:Message): return [Attachment(a, self._client) for a in self.get('attachments', [])]

In [ ]:
atts = msg.attachments; atts

[Attachment(filename='README.md', size=28163, type=text/markdown; charset=utf-8)]

In [ ]:
readme = (await atts[0].fetch()).decode()
print(readme[:16])

# cordslite 🍺




DMs (Direct Messages) are just regular channels in Discord's API — no special handling needed! To start a DM conversation, POST to `/users/@me/channels` with a `recipient_id`. Discord returns a standard channel object, so `send()` and `messages()` work exactly as they do for guild channels.

To detect DMs in the gateway, check for the absence of `guild_id` — DM messages don't belong to any guild, so this field is missing or `None`. This makes it easy to route DM vs guild messages in your bot's handler.

In [ ]:
#| export
@patch
async def create_dm(self:DiscordClient, user_id):
    r = await self._req('POST', '/users/@me/channels', recipient_id=user_id)
    return Channel(r, self)

In [ ]:
# # Commented out so we don't spam Nate
# dm = await dc.create_dm('346450717025894400')  # nathan's user ID
# await dm.send('Hello from DMs!')

Members vs Users: A **User** is a global Discord account. A **Member** is a user *within a specific guild*—it has guild-specific data like nickname, roles, and join date. The `nick or user['username']` pattern shows the server nickname if set, otherwise falls back to the global username.

Note: The members endpoint requires the `GUILD_MEMBERS` privileged intent enabled in your bot settings on the Developer Portal!

In [ ]:
#| export
class User(DiscordObject):
    def __repr__(self): return f"User(id={self.id}, username={self.username!r})"

class Member(DiscordObject):
    def __repr__(self):
        name = self.nick or self.user['username']
        return f"Member(id={self.user['id']}, name={name!r}, joined={self.joined_at[:10]})"

class Members(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Joined", "Roles"), lambda m: (m.user['id'], m.nick or m.user['username'], m.joined_at[:10], len(m.roles)))

@patch
async def members(self:Guild, limit=100):
    data = await self('GET', f'/guilds/{self.id}/members', limit=limit)
    return self.coll('Member', data)

In [ ]:
mems = await gld.members(5); mems

ID,Name,Joined,Roles
206121263213576192,rensdimmendaal,2026-04-14,0
281701548919095297,frax___,2026-04-14,0
346450717025894400,nate.dawgg,2026-04-14,0
660097403046723594,jeremyhoward,2026-04-14,0
736643719675248691,tommyc.xyz,2026-04-14,0


In [ ]:
#| export
def lbl(ch):
    l = "#" if ch.type==0 else "🔊"
    l += ch.name
    if t := getattr(ch, 'topic', ''): l += ': ' + t
    return l

@patch
async def tree(self:Guild, include_members=True, member_limit=1000):
    by_parent,cats = defaultdict(list),[None]
    for c in await self.channels():
        if c.type == 4: cats.append(c)
        elif c.type in (0,2): by_parent[c.parent_id].append(c)
    
    lines = [f"{self.name} [{self.id}]"]
    for cat in cats:
        lines.append(f"|-- {getattr(cat, 'name', 'Uncategorized')}")
        for ch in by_parent[getattr(cat, 'id', None)]:
            lines.append(f"|   |-- {lbl(ch)} [{ch.id}]")

    if include_members:
        lines.append("|-- Members")
        for m in await self.members(member_limit):
            u = m.user
            name = m.nick or u.get('global_name') or u['username']
            lines.append(f"|   |-- {name} [{u['id']}]")

    return "\n".join(lines)

In [ ]:
print(await gld.tree())

AAI-test-dev [1493461895615873044]
|-- Uncategorized
|-- Text Channels
|   |-- #general [1493461896139903028]
|-- Voice Channels
|   |-- 🔊General [1493461896139903029]
|-- Members
|   |-- Rens [206121263213576192]
|   |-- curtis [281701548919095297]
|   |-- natedog [346450717025894400]
|   |-- Jeremy Howard [660097403046723594]
|   |-- Tommy [736643719675248691]
|   |-- share2self [1274262434152185857]


In [ ]:
#| export
@patch
async def search_all(self:Guild, limit=500, delay=1.0, max_age_days=None, show=False, **kwargs):
    "Paginated search returning up to `limit` messages"
    all_msgs, offset = [], 0
    while len(all_msgs) < limit:
        msgs = await self.search(offset=offset, limit=25, **kwargs)
        if not msgs: break
        if max_age_days is not None:
            cutoff = datetime.now(timezone.utc) - timedelta(days=max_age_days)
            msgs = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
        all_msgs.extend(msgs)
        offset += 25
        if show: print(f'Fetched {len(msgs)} more')
        if len(msgs) < 25: break
        await asyncio.sleep(delay)
    return Messages(all_msgs[:limit])

@patch
@delegates(Guild.search_all, but=['channel_id'])
async def search_all(self:Channel, **kwargs):
    gld = await self.guild
    return await gld.search_all(channel_id=self.id, **kwargs)

In [ ]:
# r = await ch.search_all()
# len(r)

In [ ]:
#| export
@patch
async def delete(self:Message):
    "Delete this message"
    return await self('DELETE', f'/channels/{self.channel_id}/messages/{self.id}')

@patch
async def delete_message(self:Channel, message_id):
    "Delete a message by ID"
    return await self('DELETE', f'/channels/{self.id}/messages/{message_id}')

@patch
async def bulk_delete(self:Channel, message_ids):
    "Bulk delete messages (must be <14 days old)"
    return await self('POST', f'/channels/{self.id}/messages/bulk-delete', messages=[str(i) for i in message_ids])

In [ ]:
#| export
@patch
async def create_thread(self:Message, name):
    "Create a thread from this message"
    r = await self('POST', f'/channels/{self.channel_id}/messages/{self.id}/threads', name=name)
    return Channel(r, self)

@patch
async def thread(self:DiscordClient, thread_id):
    "Fetch a thread (which is a Channel)"
    return Channel(await self._req('GET', f'/channels/{thread_id}'), self)

In [ ]:
#| export
@patch
async def _del_rotating(self:Channel, message_ids, delay=0.5, show=False):
    if show: print(f'Deleting {len(message_ids)} messages...')
    for i,mid in enumerate(message_ids):
        use_user = i % 2 == 1 and bool(self._client.user_token)
        try: await self('DELETE', f'/channels/{self.id}/messages/{mid}', use_user=use_user)
        except DiscordError as e:
            if e.args[0] == 10008: print(f"Already deleted: {mid}"); continue
            raise
        await asyncio.sleep(delay)
        if show: print('.', end='', flush=True)
    return len(message_ids)

In [ ]:
#| export
@patch
async def search_and_delete_all(self:Channel, content, delay=2, show=False, **kwargs):
    "Bulk delete recent msgs, individually delete older ones"
    assert content, "Must include content to search for"
    cutoff = datetime.now(timezone.utc) - timedelta(days=13)
    gld = await self.guild
    msgs = await gld.search_all(content=content, channel_id=self.id, show=show, **kwargs)
    seen = set()
    msgs = [m for m in msgs if m.id not in seen and not seen.add(m.id)]
    recent = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
    old = [m for m in msgs if datetime.fromisoformat(m.timestamp) <= cutoff]
    for i in range(0, len(recent), 20):
        ms = recent[i:i+20]
        if len(ms)>1: await self.bulk_delete([m.id for m in ms])
        else: await self._del_rotating([m.id for m in ms], show=show)
        if show: print(f'Bulk deleted {len(ms)}')
        await asyncio.sleep(delay)
    if old: await self._del_rotating([m.id for m in old], show=show, delay=delay)
    return len(msgs)

Webhooks let external services post to a channel without a bot token—executing one only needs its `id` and `token`. We wrap them in a `Webhook` class, with `webhooks` listings on both `Channel` and `Guild`, and `edit`/`delete`/`send` on the webhook itself. `send` uses `?wait=true` so Discord returns the created message, and supports per-message `username`/`avatar_url` overrides.

In [ ]:
#| export
class Webhook(DiscordObject):
    def __repr__(self): return f"Webhook(id={self.id}, name={self.name!r})"

class Webhooks(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Channel"), lambda w: (w.id, w.name, w.channel_id))

In [ ]:
#| export
@patch
async def webhooks(self:Channel):
    "List this channel's webhooks"
    return self.coll('Webhook', await self('GET', f'/channels/{self.id}/webhooks'))

@patch
async def webhooks(self:Guild):
    "List all webhooks in this guild"
    return self.coll('Webhook', await self('GET', f'/guilds/{self.id}/webhooks'))

@patch
async def webhook(self:DiscordClient, webhook_id):
    "Fetch a webhook by ID"
    return Webhook(await self._req('GET', f'/webhooks/{webhook_id}'), self)

In [ ]:
#| export
@patch
async def create_webhook(self:Channel, name):
    "Add webhook `name` to Channel"
    r = await self('POST', f'/channels/{self.id}/webhooks', name=name)
    return Webhook(r, self)

In [ ]:
#| export
@patch
async def edit(self:Webhook, name=None, channel_id=None):
    "Modify this webhook's `name` or move it to `channel_id`"
    return Webhook(await self('PATCH', f'/webhooks/{self.id}', name=name, channel_id=channel_id), self._parent)

@patch
async def delete(self:Webhook):
    "Delete this webhook"
    return await self('DELETE', f'/webhooks/{self.id}')

@patch
async def send(self:Webhook, content='', username=None, avatar_url=None):
    "Execute this webhook, optionally overriding the display `username`/`avatar_url`"
    r = await self('POST', f'/webhooks/{self.id}/{self.token}?wait=true', content=content, username=username, avatar_url=avatar_url)
    return Message(r, self)

## Gateway API

The Gateway is Discord's WebSocket connection for **real-time events**. Unlike the REST API (request/response), the Gateway pushes events to you as they happen—new messages, reactions, user joins, etc.

The connection lifecycle is:
1. Fetch WSS URL from `/gateway/bot`
2. Connect to WebSocket
3. Receive **Hello** event with heartbeat interval
4. Start heartbeat loop to keep connection alive
5. Send **Identify** with your token and intents
6. Receive **Ready** event—you're connected!

**Intents** tell Discord which events you want. They're bitwise flags you OR together:
- `1 << 0` = GUILDS (guild/channel events)
- `1 << 9` = GUILD_MESSAGES (message events)
- `1 << 15` = MESSAGE_CONTENT (privileged—see message content)

In [ ]:
#| export
class GatewayClient:
    def __init__(self, intents, client, token=None):
        self.intents,self.dc = intents,client
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.ws = self.hb_int = self.session_id = self.seq = None
        self.running = False
        self._tries = 0
        gw_info = httpx.get(f'{client.base_url}/gateway/bot', headers={'Authorization': f'Bot {self.token}'}).json()
        if 'url' not in gw_info: raise ConnectionError(f"Gateway auth failed: {gw_info.get('message', gw_info)}")
        self.url = f"{gw_info['url']}?v=10&encoding=json"
        self.handlers = {'READY': self.on_rdy}
    
    async def on_rdy(self, evt):
        self.session_id, self.user_id, = evt['session_id'], evt['user']['id']
        self.resume_url = evt['resume_gateway_url'] + '?v=10&encoding=json' if 'resume_gateway_url' in evt else self.url
        
    def __repr__(self): return f"GatewayClient({self.intents=}, {self.url=})"

In [ ]:
# For now, let's use basic intents for guilds and messages
intents = (1 << 0) | (1 << 7) | (1 << 9) | (1 << 15) # GUILDS | VOICE_EVENTS | GUILD_MESSAGES | MESSAGE_CONTENT

gc = GatewayClient(intents, dc); gc

GatewayClient(self.intents=33409, self.url='wss://gateway.discord.gg?v=10&encoding=json')

`Op` is a thin helper for constructing Gateway opcodes. Discord's Gateway expects JSON payloads with an `op` field (operation code) and `d` field (payload data). Rather than hand-building dicts each time, `Op.identify()` and `Op.heartbeat()` return properly structured payloads ready to send over the WebSocket.

In [ ]:
#| export
class Op(AttrDict):
    def __repr__(self): return f"Op(op={self.op}, d={self.d})"
    @classmethod
    def identify(cls, token, intents):
        d = AttrDict(token=token, intents=intents, properties=dict(os='linux',browser='discord_wrapper', device='discord_wrapper'))
        return cls(op=2, d=d)
    @classmethod
    def heartbeat(cls, seq): return cls(op=1, d=seq)
    @classmethod
    def resume(cls, token, session_id, seq): return cls(op=6, d=AttrDict(token=token, session_id=session_id, seq=seq))

The Gateway sends events as JSON with four fields:
- **`op`** — operation code: `0` = dispatch (real events), `1` = heartbeat request, `10` = hello, `11` = heartbeat ACK
- **`t`** — event type name (only for dispatches): `MESSAGE_CREATE`, `GUILD_CREATE`, etc.
- **`s`** — sequence number (only for dispatches): increments per event, needed for heartbeats and resuming
- **`d`** — the payload data

We track the latest `s` value so heartbeats can tell Discord "I've received everything up to here." The `Event` class wraps raw JSON and auto-converts known dispatch types (like `MESSAGE_CREATE`) into their corresponding wrapper classes (`Message`, `Channel`, etc.) via `evt_typs`.

In [ ]:
#| export
evt_typs = {'MESSAGE_CREATE': Message,
            'MESSAGE_UPDATE': Message,
            'MESSAGE_DELETE': Message,
            'GUILD_CREATE': Guild,
            'CHANNEL_CREATE': Channel}

class Event(DiscordObject):
    def __init__(self, data, client):
        super().__init__(data, client)
        typ = evt_typs.get(self.t)
        self.d = typ(self.d, client) if typ else self.d
    @property
    def type(self): return self.t
    @property
    def seq(self): return self.s
    def __repr__(self): return f"Event({self.op=}, {self.type=}, {self.seq=})"

In [ ]:
#| export
@patch
async def send(self:websockets.asyncio.client.ClientConnection, msg, **kw):
    if isinstance(msg, dict): msg = json.dumps(msg)
    return await self._orig_send(msg, **kw)

`recv_evt` is the low-level building block for receiving events. It reads one raw WebSocket message, wraps it as an `Event` (auto-converting known dispatch types), and updates the sequence counter. You can use it directly to pull events one at a time — useful for debugging or understanding what Discord is sending.

In [ ]:
#| export
@patch
async def recv_evt(self:GatewayClient):
    evt = Event(json.loads(await self.ws.recv()), self.dc)
    if evt.op == 0 and evt.seq: self.seq,self._tries = evt.seq,0
    return evt

Rather than special-casing the READY event inside `_listen`, we register it as a regular dispatch handler via `gc.on()`. When Discord confirms a successful Identify, `on_rdy` caches `session_id`, `user_id`, and `resume_gateway_url` — the three values needed to Resume after a disconnect. Note the query params appended to `resume_gateway_url` — Discord returns it bare, but the docs require the same version and encoding as the initial connection.

In [ ]:
#| export
@patch
def on(self:GatewayClient, event_type, handler): self.handlers[event_type] = handler

The gateway runs on two cooperating loops. `_hb` sends heartbeats at the interval Discord specifies and watches for ACK responses — if one is missed, the connection is zombied and gets closed with code 4000 (which preserves the session for Resuming). `_listen` reads events and dispatches them by opcode. On disconnect (exception from `recv_evt`), it calls `_reconnect` to swap in a fresh WebSocket. The new connection triggers Hello (op 10), which restarts the heartbeat and sends either Identify (first connect) or Resume (reconnect) depending on whether a session already exists. Op 7 (Reconnect) follows the same path — close and reopen with a resume.

All reconnection funnels through `_reconnect`. A resume closes with code 4000 (keeping the session valid) and opens a new socket to `resume_gateway_url`; a re-identify deliberately closes with 1000 (invalidating the session) and goes back to the original gateway URL. The listener's op 10 handler then takes over automatically.

In [ ]:
#| export
@patch
async def _hb(self:GatewayClient):
    await asyncio.sleep(self.hb_int / 1_000 * random.random()) # jitter
    while self.running:
        self._got_ack = False
        try: await self.ws.send(Op.heartbeat(self.seq))
        except Exception: return
        await asyncio.sleep(self.hb_int / 1_000)
        if not self._got_ack: return await self.ws.close(code=4000)

In [ ]:
#| export
@patch
async def _reconnect(self:GatewayClient, resume=False):
    resume = resume and bool(self.session_id)
    code, url = (4000, self.resume_url) if resume else (1000, self.url)
    if not resume: self.session_id = None
    try: await self.ws.close(code=code)
    except Exception: pass
    while self.running:
        await asyncio.sleep(min(2 ** self._tries, 60))
        self._tries += 1
        try:
            self.ws = await websockets.connect(url, max_size=None)
            return
        except OSError as e: print('gateway reconnect failed:', repr(e))

In [ ]:
#| export
@patch
async def _listen(self:GatewayClient):
    while self.running:
        try:
            evt = await self.recv_evt()
            if self.debug: print('DEBUG: Received Opcode:', evt.op)
            if evt.op == 10:
                self.hb_int = evt.d['heartbeat_interval']
                if hasattr(self, '_hb_task') and self._hb_task: self._hb_task.cancel()
                self._hb_task = asyncio.create_task(self._hb())
                if self.session_id: await self.ws.send(Op.resume(self.token, self.session_id, self.seq))
                else: await self.ws.send(Op.identify(self.token, self.intents))
            elif evt.op == 0 and evt.type in self.handlers: asyncio.create_task(self.handlers[evt.type](evt.d))
            elif evt.op == 11: self._got_ack = True
            elif evt.op == 1: await self.ws.send(Op.heartbeat(self.seq))
            elif evt.op == 7: await self._reconnect(resume=True)
            elif evt.op == 9: await self._reconnect(resume=evt.d)
        except websockets.exceptions.ConnectionClosed as e:
            if not self.running: break
            print(f'gateway closed: {e.code} {e.reason}')
            if e.code in (4004, 4010, 4011, 4012, 4013, 4014): # fatal: bad token/shard/intents
                self.running = False
                break
            await self._reconnect(resume=e.code not in (4007, 4009))
            continue
        except Exception as e:
            print('gateway listen error:', repr(e))
            await self._reconnect(resume=True)
            continue

The reconnect rules come straight from the [gateway docs](https://docs.discord.com/developers/topics/gateway#disconnections): opcode 7 and unexpected closes should *resume* (re-identifying burns the 1,000/day identify budget and drops queued events), close 4007/4009 mean the session is unrecoverable so re-identify, and 4004/4010–4014 (bad token, bad shard, bad intents) are fatal — retrying would loop forever. Closing with 1000/1001 tells Discord to invalidate the session, so `_reconnect` only uses 1000 when abandoning it on purpose. Failed connects retry with exponential backoff (reset whenever a dispatch arrives), and `max_size=None` stops the websockets library from killing the connection when a large guild's GUILD_CREATE exceeds its 1MB default. We test the close-code policy with a fake socket that replays a payload then dies with a chosen close code, recording what `_reconnect` was asked to do.

In [0]:
from websockets.frames import Close

class _GWS:
    "Yields queued gateway payloads, then raises ConnectionClosed(`code`)."
    def __init__(self, msgs, code=1006): self.msgs,self.code = list(msgs),code
    async def recv(self):
        if self.msgs: return json.dumps(self.msgs.pop(0))
        raise websockets.exceptions.ConnectionClosedError(Close(self.code, ''), None)
    async def close(self, code=1000): pass
    async def send(self, msg): pass

def _mk_gc(ws):
    tgc = object.__new__(GatewayClient)
    tgc.dc,tgc.token,tgc.intents = None,'t',0
    tgc.ws,tgc.hb_int,tgc.session_id,tgc.seq = ws,None,'s1',42
    tgc.running,tgc.debug,tgc._hb_task,tgc._tries,tgc.handlers = True,False,None,0,{}
    tgc.rec = []
    async def _rec(resume=False): tgc.rec.append(resume); tgc.running = False
    tgc._reconnect = _rec
    return tgc

In [0]:
tgc = _mk_gc(_GWS([dict(op=7, s=None, t=None, d=None)]))  # RECONNECT request: resume
await tgc._listen()
assert tgc.rec == [True]

tgc = _mk_gc(_GWS([], code=1006))  # network drop / zombied connection: resume
await tgc._listen()
assert tgc.rec == [True]

tgc = _mk_gc(_GWS([], code=4007))  # invalid resume seq: re-identify
await tgc._listen()
assert tgc.rec == [False]

tgc = _mk_gc(_GWS([], code=4004))  # bad token: give up rather than loop
await tgc._listen()
assert tgc.rec == [] and not tgc.running

`start` opens the initial WebSocket and spawns the listener — nothing else. The listener handles Hello, kicks off the heartbeat, and sends Identify, so the full connection lifecycle is driven by events rather than imperative setup.

In [ ]:
#| export
@patch
async def start(self:GatewayClient, debug=False):
    self.debug = debug
    self.running = True
    self.ws = await websockets.connect(self.url, max_size=None)
    self._listen_task = asyncio.create_task(self._listen())

In [ ]:
await gc.start(debug=True)

DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DBuddy: Test our event listener! Otters are awesome 🦦


sent 4000 (private use); then received 4000 (private use)


DEBUG: Received Opcode: 10
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 10


sent 1000 (OK); then received 1000 (OK)


DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 0
DBuddy: Did we re-identify?


Let's now test that our gateway is getting events. Here is a simple handlers for new messages.

In [ ]:
async def on_msg(msg):
    if msg.get('guild_id') != gid: return
    if not msg.author.get('bot'): return
    print(f"{msg.author['username']}: {msg.content}")

gc.on('MESSAGE_CREATE', on_msg)

Now, we can send a message and it should be printed out.

In [ ]:
await ch.send('Test our event listener! Otters are awesome 🦦')

<div class="prose" markdown="1">

```python
Message(id=1507463593162047658, author='DBuddy', content='Test our event listener! Otter…')
```

</div>

Let's trigger and reconnection that resumes and one that doesn't and see if everything works correctly and we still end up with our printed message.

In [ ]:
await gc._reconnect(resume=True)
await asyncio.sleep(2)
await ch.send('Houston, do we have a problem?')

<div class="prose" markdown="1">

```python
Message(id=1507463611893809202, author='DBuddy', content='Houston, do we have a problem?')
```

</div>

In [ ]:
await gc._reconnect(resume=False)
await asyncio.sleep(2)
await ch.send('Did we re-identify?')

<div class="prose" markdown="1">

```python
Message(id=1507463630873039008, author='DBuddy', content='Did we re-identify?')
```

</div>

`stop` is a full shutdown — cancels both background tasks and closes the WebSocket with the default code (1000), which tells Discord to invalidate the session. After this, a fresh `start()` would need to Identify from scratch.

In [ ]:
#| export
@patch
async def stop(self:GatewayClient):
    self.running = False
    for t in (self._hb_task, self._listen_task):
        if t: t.cancel()
    if self.ws: await self.ws.close()

In [ ]:
await gc.stop()

## Voice API

In [ ]:
await gc.start(debug=True)
vch = chs[3]; vch

Voice requires **three simultaneous connections**:

1. **Main Gateway WebSocket** (`gc`) — to request joining a voice channel and receive voice server info
2. **Voice Gateway WebSocket** — a *separate* WebSocket to a dedicated voice server for session coordination
3. **Voice UDP** — a UDP (datagram) connection for actual audio data. UDP is used over TCP because real-time audio needs low latency more than guaranteed delivery

`VoiceClient` manages the voice gateway WebSocket and UDP connections for a single voice channel, coordinated through the main `GatewayClient`.

In [ ]:
#| export
class VoiceClient:
    def __init__(self, gc, gid, ch):
        store_attr()
        self.state_ready = asyncio.Event()
        self.server_ready = asyncio.Event()
        self.rdy = asyncio.Event()
        self.sess = asyncio.Event()

        self.session_id = self.token = self.endpoint = None
        self.ws = self.udp = None
        self._listen_task = self._hb_task = self._ka_task = None
        self.v_seq = -1
        self._tries = 0
        self.resuming = self._rejoining = False
        self.resumed = asyncio.Event()

        self.ssrc_to_user = {}
        self.decoders = {}
        self.dave_pending_transitions = {}
        self.dave_session = None
        self.dave_protocol_version = 0

        self.gc.on("VOICE_STATE_UPDATE", self.on_state_update)
        self.gc.on("VOICE_SERVER_UPDATE", self.on_server_update)
    
    async def on_state_update(self, data):
        if data["user_id"] != self.gc.user_id: return
        self.session_id = data["session_id"]
        ch = data.get("channel_id")
        if ch is None:
            if getattr(self, 'running', False) and not self._rejoining:
                print('removed from voice channel; disconnecting')
                await self.disconnect()
            return
        if ch != self.ch.id: self.ch['id'] = ch # moved to another channel
        self.state_ready.set()

    async def on_server_update(self, data):
        if data["guild_id"] != self.gid: return
        tok,ep = data["token"],data["endpoint"]
        if ep and ep.startswith("wss://"): ep = ep[6:]
        moved = self.server_ready.is_set() and (ep,tok) != (self.endpoint,self.token)
        self.token,self.endpoint = tok,ep
        if not ep: return self.server_ready.clear() # server went away; a follow-up update names the new one
        self.server_ready.set()
        if moved and getattr(self, 'running', False):
            asyncio.create_task(self._reconnect(resume=False, rejoin=False))

    def __repr__(self): return f'VoiceClient({self.ch=})'

In [ ]:
vc = VoiceClient(gc, gid, vch); vc

VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2))

Voice requires additional `Op` opcodes beyond the main gateway's identify/heartbeat. These handle the voice-specific protocol: requesting to join a channel (`voice_state`), authenticating with the voice server (`voice_identify`), negotiating encryption (`select_protocol`), signaling audio intent (`speaking`), and keeping the voice connection alive (`voice_heartbeat`).

The voice heartbeat uses a different format from the main gateway — v8 requires `seq_ack` tracking the last received sequence number. It must start **immediately** after receiving Hello, before UDP setup, or the voice WebSocket will time out (close code 4006).

In [ ]:
#| export
@patch(cls_method=True)
def voice_state(cls:Op, guild_id, channel_id):
    return cls(op=4, d=AttrDict(guild_id=guild_id, channel_id=channel_id, self_mute=False, self_deaf=False))
@patch(cls_method=True)
def voice_identify(cls:Op, server_id, user_id, session_id, token):
    return cls(op=0, d=AttrDict(server_id=server_id, user_id=user_id,
                                session_id=session_id, token=token,
                                max_dave_protocol_version=davey.DAVE_PROTOCOL_VERSION))
@patch(cls_method=True)
def select_protocol(cls:Op, ip, port, mode='aead_xchacha20_poly1305_rtpsize'):
    return cls(op=1, d=AttrDict(protocol='udp', data=dict(address=ip, port=port, mode=mode)))
@patch(cls_method=True)
def speaking(cls:Op, ssrc, speaking=0):
    return cls(op=5, d=AttrDict(speaking=speaking, delay=0, ssrc=ssrc))
@patch(cls_method=True)
def voice_heartbeat(cls:Op, seq_ack=-1):
    return cls(op=3, d=AttrDict(t=int(time.time() * 1000), seq_ack=seq_ack))
@patch(cls_method=True)
def voice_resume(cls:Op, server_id, session_id, token, seq_ack=-1):
    return cls(op=7, d=AttrDict(server_id=server_id, session_id=session_id, token=token, seq_ack=seq_ack))

In [ ]:
#| export
@patch
async def _join(self:VoiceClient, timeout=10):
    self.state_ready.clear()
    self.server_ready.clear()
    await self.gc.ws.send(Op.voice_state(self.gid, self.ch.id))
    await asyncio.wait_for( asyncio.gather(self.state_ready.wait(), self.server_ready.wait()),
                            timeout)

In [ ]:
# await vc._join()

Op 2 - Ready Event

**IP Discovery**: Discord needs your *external* IP/port (what the internet sees after NAT), not your local one. We send a 74-byte UDP request containing our SSRC, and Discord responds with our external address filled in:

| Field | Size | Description |
|-------|------|-------------|
| Type | 2 bytes | 1=request, 2=response |
| Length | 2 bytes | 70 (size of remaining fields) |
| SSRC | 4 bytes | Our audio stream identifier |
| Address | 64 bytes | Blank in request; our external IP in response |
| Port | 2 bytes | Blank in request; our external port in response |

Note: Discord requires a **Speaking** event (even with `speaking=0`) before it will send audio packets to you.

In [ ]:
#| export
async def _ip(trans, proto, ssrc):
    pkt = bytearray(74)
    pkt[0:2],pkt[2:4],pkt[4:8] = (1).to_bytes(2,'big'),(70).to_bytes(2,'big'),ssrc.to_bytes(4,'big')
    trans.sendto(pkt)
    r = await proto.packets.get()
    return r[8:72].decode('utf-8').rstrip('\x00'), int.from_bytes(r[72:74], 'big')

class VoiceUDP(asyncio.DatagramProtocol):
    def __init__(self): self.packets = asyncio.Queue()
    def datagram_received(self, data, addr): self.packets.put_nowait(data)

@patch
async def _keepalive(self:VoiceClient, interval=5):
    "Ping the voice UDP socket so NAT mappings survive listen-only sessions"
    n = 0
    while self.running:
        try: self.trans.sendto(n.to_bytes(8, 'big'))
        except Exception: pass
        n = (n + 1) & 0xffffffffffffffff
        await asyncio.sleep(interval)

@patch
async def _udp(self:VoiceClient):
    self.loop = asyncio.get_running_loop()
    self.trans, self.proto = await self.loop.create_datagram_endpoint(VoiceUDP, remote_addr=(self.vip, self.vport))
    self.ip, self.port = await _ip(self.trans, self.proto, self.ssrc)
    if self._ka_task: self._ka_task.cancel()
    self._ka_task = asyncio.create_task(self._keepalive())
    await self.ws.send(Op.select_protocol(self.ip, self.port))

@patch
async def _handle_rdy(self:VoiceClient, d):
    self.ssrc,self.vip,self.vport,self.modes = d["ssrc"],d["ip"],d["port"],d["modes"]
    self.rdy.set()
    await self._udp()

One more thing the UDP path needs: while a bot only *listens* (like a recorder), it never sends a packet after IP discovery, so NAT/conntrack mappings expire after a few minutes and inbound audio silently stops while the websocket stays healthy. A tiny 8-byte counter ping every 5s (the same trick discord.js uses) keeps the mapping alive; the voice server ignores unknown packets.

In [0]:
fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))
pings = []
fvc.trans = SimpleNamespace(sendto=pings.append)
fvc.running = True
_t = asyncio.create_task(fvc._keepalive(interval=0.05))
await asyncio.sleep(0.22)
fvc.running = False
_t.cancel()
assert len(pings) >= 4 and pings[:2] == [(0).to_bytes(8, 'big'), (1).to_bytes(8, 'big')]

Op 4 - Session Description

In [ ]:
#| export
spf = 960
sr = 48_000
n_chs = 2

@patch
async def _dave(self:VoiceClient):
    if self.dave_version > 0:
        if hasattr(self, 'dave'): self.dave.reinit(self.dave_version, int(self.gc.user_id), int(self.ch.id))
        else: self.dave = davey.DaveSession(self.dave_version, int(self.gc.user_id), int(self.ch.id))
        await self.ws.send(bytes([26]) + self.dave.get_serialized_key_package())
    elif getattr(self, 'dave', None):
        self.dave.reset()
        self.dave.set_passthrough_mode(True, 10)

@patch
async def _handle_sess_desc(self:VoiceClient, d):
    self.mode,self.secret_key = d["mode"],bytes(d["secret_key"])
    self.dave_version = d.get("dave_protocol_version", 0)
    await self._dave()
    await self.ws.send(Op.speaking(self.ssrc, 0))
    self.send_seq = random.randrange(0, 65536)
    self.send_ts = random.randrange(0, 2**32)
    self.send_nonce = 0
    self.encoder = opuslib_next.Encoder(sr, n_chs, opuslib_next.APPLICATION_AUDIO)
    self._tries = 0
    self.sess.set()

In [ ]:
#| export
@patch
def execute_transition(self:VoiceClient, tid):
    if tid not in self.dave_pending_transitions: return
    self.dave_version = self.dave_pending_transitions.pop(tid)

@patch
async def _handle_trans(self:VoiceClient, op, d):
    if op == 21:  # DAVE_PREPARE_TRANSITION
        tid = d["transition_id"]
        self.dave_pending_transitions[tid] = d["protocol_version"]
        if tid == 0: self.execute_transition(tid)
        else:
            if d["protocol_version"] == 0 and self.dave:
                self.dave.set_passthrough_mode(True, 120)
            await self.ws.send({"op": 23, "d": {"transition_id": tid}})
    elif op == 22:  # DAVE_EXECUTE_TRANSITION
        self.execute_transition(d["transition_id"])
    elif op == 24:  # DAVE_PREPARE_EPOCH
        if d["epoch"] == 1:
            self.dave_version = d["protocol_version"]
            await self._dave()
    else: print("voice json", op, d)

@patch
async def _handle_mls(self:VoiceClient, msg):
    self.v_seq = int.from_bytes(msg[:2], "big")
    op,data = msg[2],msg[3:]

    if op == 25: self.dave.set_external_sender(data)
    elif op == 27:
        typ = data[0]
        res = self.dave.process_proposals(davey.ProposalsOperationType.append if typ == 0
                                                  else davey.ProposalsOperationType.revoke, data[1:])
        if isinstance(res, davey.CommitWelcome):
            await self.ws.send(bytes([28]) + res.commit + (res.welcome or b""))
    elif op == 29:
        tid = int.from_bytes(data[:2], "big")
        self.dave.process_commit(data[2:])
        if tid:
            self.dave_pending_transitions[tid] = self.dave_version
            await self.ws.send({"op": 23, "d": {"transition_id": tid}})
    elif op == 30:
        tid = int.from_bytes(data[:2], "big")
        self.dave.process_welcome(data[2:])
        if tid:
            self.dave_pending_transitions[tid] = self.dave_version
            await self.ws.send({"op": 23, "d": {"transition_id": tid}})
    else: print("voice bin", self.v_seq, op, len(msg))

The voice gateway runs on two cooperating loops. `_hb` sends heartbeats at the interval Discord specifies and watches for ACK responses; a missed ACK closes with code 4000, which preserves the session for a resume attempt. `_reconnect` uses one path for both flavors: a resume keeps `session_id`, `token`, `endpoint`, UDP, and DAVE state intact, while a fresh reconnect resets the audio session and re-runs `_join` to get new server info from the main gateway. Because `_listen` schedules `_reconnect` as its own task, `_reconnect` must never cancel the task it is running in (that was a bug that killed reconnects mid-flight), so it only cancels `_listen_task` when called from elsewhere. Failed connection attempts retry with exponential backoff (capped at 30s), and the counter resets once a session is established or resumed. Recording state is preserved in either path, so per-user files keep accumulating across reconnects with only a silence gap during the swap.

In [ ]:
#| export
@patch
async def _hb(self:VoiceClient):
    await asyncio.sleep(self.hb_int * random.random()) # jitter
    while self.running:
        self._got_ack = False
        try: await self.ws.send(Op.voice_heartbeat(self.v_seq))
        except Exception: return
        await asyncio.sleep(self.hb_int)
        if not self._got_ack: return await self.ws.close(code=4000)

@patch
async def _open_ws(self:VoiceClient):
    self.ws = await websockets.connect(f"wss://{self.endpoint}/?v=8", max_size=None)
    self._listen_task = asyncio.create_task(self._listen())

@patch
def reset_audio(self:VoiceClient):
    self.decoders,self.last_ts,self.ssrc_to_user,self.dave_pending_transitions = {},{},{},{}

@patch
async def _reconnect(self:VoiceClient, resume=True, rejoin=None):
    if rejoin is None: rejoin = not resume
    self.resuming = resume
    if resume: self.resumed.clear()
    t = self._listen_task
    if t and t is not asyncio.current_task(): t.cancel()
    if self._hb_task: self._hb_task.cancel()
    if self.ws:
        try: await self.ws.close(code=4000 if resume else 1000)
        except Exception: pass
    if not resume:
        if getattr(self, 'trans', None): self.trans.close()
        self.reset_audio()
        self.v_seq = -1
        self.sess.clear(); self.rdy.clear()
    if rejoin:
        self._rejoining = True
        self.state_ready.clear(); self.server_ready.clear()
        await self.gc.ws.send(Op.voice_state(self.gid, None))
        await asyncio.sleep(0.5)
        await self._join()
        self._rejoining = False
    while self.running:
        await asyncio.sleep(min(2 ** self._tries, 30))
        self._tries += 1
        try: return await self._open_ws()
        except OSError as e: print('voice reconnect failed:', repr(e))

@patch
async def _wait_dave_ready(self:VoiceClient, timeout=5):
    start = time.time()
    while self.dave_version and not self.dave.ready:
        if time.time() - start > timeout: raise TimeoutError("DAVE not ready after resume")
        await asyncio.sleep(0.05)

The listener is where the voice gateway's state machine actually runs. `HELLO` (op 8) branches on `self.resuming` to choose between `voice_resume` and `voice_identify`, which is how a single reconnect path supports both flavors. Opcode 9 confirms resume at the gateway level, but media encryption is a separate handshake; DAVE has to catch up before outgoing audio can be encrypted again, so we wait on `_wait_dave_ready` before setting `resumed`.

When the socket closes, the close code decides the recovery per the [docs](https://docs.discord.com/developers/topics/opcodes-and-status-codes#voice-voice-close-event-codes): `4006`/`4009` mean the session is gone so we must rejoin from scratch, `4014` means we were kicked/moved (or the voice server is being swapped) so we must *not* touch this endpoint again and instead wait for gateway events, `4004` is a bad token, and anything else (like `4015`, voice server crashed) is transient so we resume. Recovery runs as a separate task so the dying listener never cancels its own reconnect. Handler errors are logged and skipped rather than silently killing the listener.

In [ ]:
#| export
@patch
async def _listen(self:VoiceClient):
    while self.running:
        try:
            msg = await self.ws.recv()
            if isinstance(msg, bytes):
                await self._handle_mls(msg)
                continue

            msg = dict2obj(json.loads(msg))
            if (seq := msg.get("seq")) is not None: self.v_seq = seq
            op, d = msg.op, msg.d
            if op == 2: await self._handle_rdy(d)
            elif op == 4: await self._handle_sess_desc(d)
            elif op == 5: # SPEAKING
                if "ssrc" in d and "user_id" in d: self.ssrc_to_user[d["ssrc"]] = int(d["user_id"])
            elif op == 6: self._got_ack = True
            elif op == 8: # HELLO
                self.hb_int = d.heartbeat_interval / 1_000
                if self._hb_task: self._hb_task.cancel()
                self._hb_task = asyncio.create_task(self._hb())
                if self.resuming: await self.ws.send(Op.voice_resume(self.gid, self.session_id, self.token, self.v_seq))
                else: await self.ws.send(Op.voice_identify(self.gid, self.gc.user_id, self.session_id, self.token))
            elif op == 9:
                self.resuming = False
                self._tries = 0
                await self._wait_dave_ready()
                self.resumed.set()
            elif op > 13: await self._handle_trans(op, d)
            else: print("voice json", op, d)
        except websockets.exceptions.ConnectionClosed as e:
            if not self.running: break
            print(f'voice gateway closed: {e.code} {e.reason}')
            if e.code in (4004, 4014): break # kicked/moved/bad auth: only gateway events can revive us
            asyncio.create_task(self._reconnect(resume=e.code not in (4006, 4009)))
            break
        except Exception as e:
            print('voice listen error:', repr(e))

We can't ask Discord to hand us specific close codes on demand, so we test the close-code policy with a fake websocket that raises `ConnectionClosed` and stubs that record which recovery steps ran: a resumable close should run `_open_ws` only, a dead session should re-run `_join` then `_open_ws`, and a 4014 should do nothing.

In [0]:
from websockets.frames import Close

class _FakeGC:
    user_id = '42'
    def on(self, *a): pass
    class ws:
        @staticmethod
        async def send(msg): pass

class _FakeWS:
    def __init__(self, code): self.exc = websockets.exceptions.ConnectionClosedError(Close(code, ''), None)
    async def recv(self): raise self.exc
    async def close(self, code=1000): pass

async def _close_scenario(code, wait=3.5):
    "Feed `_listen` a ConnectionClosed(`code`) and record which recovery steps ran."
    fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))
    calls = []
    async def _open(): calls.append('open')
    async def _join(timeout=10): calls.append('join')
    fvc._open_ws,fvc._join = _open,_join
    fvc.running,fvc.ws = True,_FakeWS(code)
    fvc._listen_task = asyncio.create_task(fvc._listen())
    await asyncio.sleep(wait)
    fvc.running = False
    return calls

In [0]:
assert await _close_scenario(4015) == ['open']           # voice server crashed: resume
assert await _close_scenario(4006) == ['join', 'open']    # session invalid: full rejoin
assert await _close_scenario(4014, wait=1.5) == []        # kicked/moved: wait for gateway events

Discord can also *move* a voice session out from under us. A mid-session `VOICE_SERVER_UPDATE` (region change, server failover) delivers a new endpoint + token: the old session is dead, so we re-identify against the new endpoint — but the gateway-level voice state is still fine, so no rejoin (`rejoin=False`). A `null` endpoint means the server was lost and a follow-up update will name the new one, so we just wait. And a `VOICE_STATE_UPDATE` for our own user tells us when an admin kicked us (`channel_id` of `None` — tear down, don't fight it) or dragged us to a different channel (follow along).

In [0]:
async def _mk_fvc():
    fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))
    fvc.recon = []
    async def _rec(resume=True, rejoin=None): fvc.recon.append(dict(resume=resume, rejoin=rejoin))
    fvc._reconnect = _rec
    fvc.running,fvc.ws = True,None
    fvc.token,fvc.endpoint = 'a','old.discord.media'
    return fvc

fvc = await _mk_fvc()          # voice server moved: re-identify against the new endpoint
fvc.server_ready.set()
await fvc.on_server_update(dict(guild_id='1', token='b', endpoint='wss://new.discord.media'))
await asyncio.sleep(0.05)
assert (fvc.endpoint,fvc.token) == ('new.discord.media','b') and fvc.recon == [dict(resume=False, rejoin=False)]

fvc = await _mk_fvc()          # endpoint null: voice server lost, wait for the follow-up update
fvc.server_ready.set()
await fvc.on_server_update(dict(guild_id='1', token='b', endpoint=None))
assert not fvc.server_ready.is_set() and not fvc.recon

fvc = await _mk_fvc()          # kicked: tear down rather than fight the server
await fvc.on_state_update(dict(user_id='42', session_id='s', channel_id=None))
assert not fvc.running

fvc = await _mk_fvc()          # moved between channels: follow along
await fvc.on_state_update(dict(user_id='42', session_id='s', channel_id='999'))
assert fvc.ch.id == '999'

In [0]:
fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))  # rejoin=False reopens the ws without a gateway rejoin
calls = []
async def _open(): calls.append('open')
async def _join(timeout=10): calls.append('join')
fvc._open_ws,fvc._join = _open,_join
fvc.running = True
await fvc._reconnect(resume=False, rejoin=False)
assert calls == ['open']

In [ ]:
#| export
@patch
async def join(self:VoiceClient, debug=False):
    self.debug = debug
    self.running = True
    await self._join()
    await self._open_ws()


In [ ]:
await vc.join(debug=True)

voice json 11 {'user_ids': ['346450717025894400']}
voice json 18 {'user_id': '346450717025894400', 'flags': 2}
voice json 20 {'user_id': '346450717025894400', 'platform': 0}


voice json 15 {'any': 100}


Audio arrives as UDP packets using the **RTP (Real-time Transport Protocol)** format. Each packet contains:
- **RTP header** (12+ bytes) — version, sequence number, timestamp, SSRC (identifies which user is speaking)
- **Encrypted payload** — Opus audio wrapped first by DAVE, then by Discord's voice transport encryption using the `secret_key`
- **Nonce** (4 bytes at end) — used for the transport decrypt

Decryption is two-stage:
- First, decrypt the Discord transport layer with `aead_xchacha20_poly1305_rtpsize`
- The cipher expects a 24-byte nonce, but only 4 bytes are transmitted — we pad with 20 zero bytes
- The unencrypted RTP header is used as AAD (Additional Authenticated Data) — it's verified but not encrypted
- For `rtpsize` mode, the AAD includes the 12-byte base header, any CSRC entries (4 bytes each, usually 0), and only the 4-byte extension *preamble* — the extension *data* is part of the encrypted payload
- After transport decryption, skip past the RTP extension data to get the DAVE-protected audio frame — its length comes from the extension preamble (a count of 32-bit words), not a fixed size, and packets without the extension bit have nothing to skip
- Then decrypt that frame with DAVE: `self.dave.decrypt(uid, davey.MediaType.audio, data)`, where `uid` comes from the SSRC → user mapping learned from voice `SPEAKING` events
- The result is the Opus payload, ready for Opus decoding

Packets with byte[1] == `0x78` are RTP voice data. Other packets (like `0xC9`) are RTCP control packets used for connection quality reporting — we skip those.


In [ ]:
#| export
@patch
def decrypt(self:VoiceClient, pkt, uid):
    csrc_cnt = pkt[0] & 0x0F
    has_ext = bool(pkt[0] & 0x10)
    hdr_sz = 12 + csrc_cnt * 4 + (4 if has_ext else 0)
    nonce = bytearray(24) # only use 4, but pad to 24 bytes as expected by the cipher
    nonce[:4] = pkt[-4:]
    d, hdr, nonce = bytes(pkt[hdr_sz:-4]), bytes(pkt[:hdr_sz]), bytes(nonce)
    d = xchacha_decrypt(d, hdr, nonce, bytes(self.secret_key))
    if has_ext: d = d[4 * int.from_bytes(pkt[hdr_sz-2:hdr_sz], 'big'):] # ext payload length is in 32-bit words
    return self.dave.decrypt(uid, davey.MediaType.audio, d) if getattr(self, 'dave', None) else d

A hardcoded 8-byte skip here used to corrupt audio whenever the extension wasn't exactly two words (and ate real Opus data on extension-less packets). We check the parsing with crafted packets round-tripped through the real cipher: varying extension sizes, no extension, and no DAVE session (where the transport plaintext *is* the Opus frame).

In [0]:
_key = bytes(range(32))
def _mk_pkt(payload, ext_words=2, has_ext=True):
    hdr = struct.pack('>BBHII', 0x80 | (0x10 if has_ext else 0), 0x78, 1, 960, 1234)
    if has_ext: hdr += struct.pack('>HH', 0xBEDE, ext_words)
    n4 = (7).to_bytes(4, 'big')
    enc = xchacha_encrypt((bytes(4 * ext_words) if has_ext else b'') + payload, hdr, n4 + b'\0'*20, _key)
    return hdr + enc + n4

fvc = object.__new__(VoiceClient)
fvc.secret_key = _key
for kw in [dict(ext_words=0), dict(ext_words=2), dict(ext_words=3), dict(has_ext=False)]:
    assert fvc.decrypt(_mk_pkt(b'opus!', **kw), 0) == b'opus!', kw

`decode` turns one incoming RTP voice packet into PCM audio, while smoothing over the messy parts of real-time UDP.

The flow is:
- Skip anything that isn't RTP voice data — Discord also sends RTCP/control packets on the same UDP path
- Read the packet's SSRC and map it to a Discord user ID using the `SPEAKING` events collected earlier
- Ignore packets from ourselves, since we don't want to record or mix our own outgoing audio back in
- Keep a separate Opus decoder per user ID, because Opus decoder state is stream-specific and cannot be safely shared across speakers
- Track RTP timestamps per SSRC so we can detect dropped packets
- For small gaps, ask Opus packet-loss concealment to synthesize missing frames with `dec.decode(b'', spf)`
- For larger gaps, insert raw silence instead of asking the decoder to invent too much audio
- Decrypt the packet into an Opus frame, then decode that frame into signed 16-bit 48 kHz stereo PCM

Decrypt errors are treated like packet loss. If transport/DAVE decrypt fails, we don't crash the receive loop — we return any gap fill plus one concealed Opus frame. UDP voice is allowed to be imperfect; the important thing is that the stream keeps moving.

In [ ]:
#| export
def silence(n_smpls:int): return b'\x00' * (n_smpls * n_chs * 2) # 2 bytes per sample (s16le)

@patch
def decode(self:VoiceClient, pkt):
    if (pkt[1] & 0x7F) != 0x78: return # skip RTCP/control packets
    ssrc = int.from_bytes(pkt[8:12], 'big')
    uid = self.ssrc_to_user.get(ssrc)
    if uid is None or uid == int(self.gc.user_id): return
    ts = int.from_bytes(pkt[4:8], 'big')
    prev = getattr(self, 'last_ts', {}).get(ssrc)
    self.last_ts = getattr(self, 'last_ts', {})

    dec = self.decoders.setdefault(uid, opuslib_next.Decoder(48000, 2))
    fill = []
    if prev is not None:
        missing = max(((ts - prev) & 0xffffffff) // spf - 1, 0)
        fill = [dec.decode(b'', spf) for _ in range(missing)] if missing <= 3 else [silence(missing * spf)]

    self.last_ts[ssrc] = ts
    try: opus = self.decrypt(pkt, uid)
    except ValueError: return b''.join(fill) + dec.decode(b'', spf)
    return b''.join(fill) + dec.decode(opus, 5760)

Recording is split per speaker by opening one ffmpeg process per user and writing that user's decoded PCM to its stdin. If someone starts speaking late, we write silence from `_rec_start` up to their first packet so all speaker files share the same timeline.

In [ ]:
#| export
def _write_silence(f, n_smpls, chunk=sr):
    "Write `n_smpls` of silence to `f` in `chunk`-sample pieces (blocking; run in a thread)"
    for i in range(0, n_smpls, chunk): f.write(silence(min(chunk, n_smpls - i)))

@patch
async def _get_proc(self:VoiceClient, uid):
    if uid not in self._rec_procs:
        path = str(self._rec_path.with_stem(f'{self._rec_path.stem}_{uid}'))
        p = ( ffmpeg.input('pipe:', f='s16le', ar=sr, ac=n_chs)
                    .output(path)
                    .overwrite_output()
                    .run_async(pipe_stdin=True, quiet=True))
        self._rec_procs[uid] = (p, path)
        await asyncio.to_thread(_write_silence, p.stdin, int((time.time() - self._rec_start) * sr))
    return self._rec_procs[uid]

@patch
async def _recv_audio(self:VoiceClient):
    while self._recording:
        try: pkt = await asyncio.wait_for(self.proto.packets.get(), timeout=0.1)
        except asyncio.TimeoutError: continue
        if not (pcm := self.decode(pkt)): continue

        ssrc = int.from_bytes(pkt[8:12], 'big')
        uid = self.ssrc_to_user.get(ssrc, ssrc)
        p, path = await self._get_proc(uid)
        await asyncio.to_thread(p.stdin.write, pcm)

These writes matter more than they look: `stdin.write` into an ffmpeg pipe *blocks* once the pipe buffer fills, and a blocked event loop stops both heartbeat tasks — so a slow disk during a long recording used to zombie the gateway and voice connections (a disconnect that only shows up mid-recording). All pipe writes now run in a thread via `asyncio.to_thread`, and the catch-up silence prefix for a speaker who first talks late in a recording (potentially hundreds of MB) is written in one-second chunks rather than one giant allocation. We test with a pipe that blocks 150ms per write while a ticker measures event-loop responsiveness: with threaded writes the ticker keeps ticking.

In [0]:
from types import SimpleNamespace

class _SlowPipe:
    def __init__(self): self.n = 0
    def write(self, b): time.sleep(0.15); self.n += 1

_slow = _SlowPipe()
fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))
fvc.proto = VoiceUDP()
fvc.decode = lambda pkt: b'\0' * 100
async def _gp(uid): return (SimpleNamespace(stdin=_slow), 'x')
fvc._get_proc = _gp
for _ in range(6): fvc.proto.packets.put_nowait(bytes(12))

ticks = []
async def _ticker():
    while True:
        ticks.append(1)
        await asyncio.sleep(0.01)
_t = asyncio.create_task(_ticker())
fvc._recording = True
_rec = asyncio.create_task(fvc._recv_audio())
await asyncio.sleep(1.0)
fvc._recording = False
await _rec
_t.cancel()
assert _slow.n == 6 and len(ticks) > 50, (_slow.n, len(ticks))  # ~12 ticks when writes blocked the loop

chunks = []
_write_silence(SimpleNamespace(write=lambda b: chunks.append(len(b))), int(2.5 * sr))
assert sum(chunks) == int(2.5 * sr) * n_chs * 2 and max(chunks) <= sr * n_chs * 2

`start_recording` drains the UDP queue first so stale packets from before the recording do not get written into the new files.

`stop_recording` has to close each ffmpeg stdin and wait for the process. That finalizes the containers; otherwise the files can look valid but be missing buffered audio at the end.


In [ ]:
#| export
@patch
def start_recording(self:VoiceClient, path='/tmp/recording.mp3'):
    while not self.proto.packets.empty(): self.proto.packets.get_nowait()

    self.decoders = {}
    self.last_ts = {}
    self._rec_path = Path(path)
    self._rec_procs = {}
    self._rec_start = time.time()
    self._recording = True
    self._rec_task = asyncio.create_task(self._recv_audio())
    return path

@patch
async def stop_recording(self:VoiceClient, mix=True, mix_path=None):
    self._recording = False

    if getattr(self, '_rec_task', None):
        self._rec_task.cancel()
        try: await self._rec_task
        except asyncio.CancelledError: pass

    speaker_paths = {}
    for uid, (p, path) in self._rec_procs.items():
        p.stdin.close()
        await asyncio.to_thread(p.wait)
        speaker_paths[uid] = path

    mixed_path = None
    if mix and speaker_paths:
        mixed_path = str(mix_path or self._rec_path.with_stem(f'{self._rec_path.stem}_mixed'))

        if len(speaker_paths) == 1:
            src = next(iter(speaker_paths.values()))
            out = ffmpeg.input(src).output(mixed_path).overwrite_output()
        else:
            inputs = [ffmpeg.input(path) for path in speaker_paths.values()]
            mixed = ffmpeg.filter(inputs, 'amix', inputs=len(inputs), duration='longest')
            out = mixed.output(mixed_path).overwrite_output()
        await asyncio.to_thread(out.run, quiet=True)

    return {'speakers': speaker_paths, 'mixed': mixed_path}

In [ ]:
vc.start_recording()

'/tmp/recording.mp3'

In [ ]:
out = await vc.stop_recording()
out

{'speakers': {346450717025894400: '/tmp/recording_346450717025894400.mp3'},
 'mixed': '/tmp/recording_mixed.mp3'}

Outgoing audio is the receive path in reverse: encode PCM to Opus, optionally wrap it with DAVE, then encrypt it with Discord's voice transport key and send it as an RTP packet.

The RTP sequence, timestamp, and nonce are advanced manually for each 20 ms frame. If those drift or repeat, Discord will hear either broken audio or nothing at all.

In [ ]:
#| export
@patch
def encode(self:VoiceClient, fr):
    opus = self.encoder.encode(fr, spf)
    return self.dave.encrypt_opus(opus) if self.dave_version else opus

@patch
def send_pkt(self:VoiceClient, payload):
    hdr = struct.pack('>BBHII', 0x80, 0x78, self.send_seq, self.send_ts, self.ssrc)
    nonce4 = self.send_nonce.to_bytes(4, 'big')
    enc = xchacha_encrypt(payload, hdr, nonce4 + b'\0'*20, bytes(self.secret_key))
    self.trans.sendto(hdr + enc + nonce4)
    self.send_seq = (self.send_seq + 1) & 0xffff
    self.send_ts = (self.send_ts + spf) & 0xffffffff
    self.send_nonce = (self.send_nonce + 1) & 0xffffffff

We must notify discord when our bot is speaking and stop speaking. When a transmission truly ends, the [docs](https://docs.discord.com/developers/topics/voice-connections#voice-data-interpolation) require five frames of Opus silence (`0xF8 0xFF 0xFE`) so receivers don't interpolate the tail of our audio into the next utterance — skipping them is audible as a garbled blip at the end of playback. Streamed audio that arrives in chunks (e.g. a realtime TTS feed) should pass `end=False` for all but the last chunk so the transmission stays continuous. The `finally` keeps Discord from leaving the bot marked as speaking if playback errors halfway through.

In [ ]:
#| export
opus_silence = b'\xf8\xff\xfe'

@patch
async def speaking(self:VoiceClient, on=True):
    await self.ws.send(Op.speaking(self.ssrc, int(on)))

@patch
def send_frame(self:VoiceClient, fr): self.send_pkt(self.encode(fr))

@patch
async def send_frames(self:VoiceClient, frames, end=True):
    "Send 20ms Opus `frames`; unless `end=False`, finish with 5 silence frames and clear the speaking flag."
    loop = asyncio.get_running_loop()
    nxt = loop.time()
    await self.speaking(True)
    try:
        for fr in frames:
            self.send_frame(fr)
            nxt += spf / sr
            await asyncio.sleep(max(0, nxt - loop.time()))
    finally:
        if end:
            for _ in range(5): # per the docs, stops unintended Opus interpolation into the next transmission
                self.send_pkt(self.dave.encrypt_opus(opus_silence) if self.dave_version else opus_silence)
                nxt += spf / sr
                await asyncio.sleep(max(0, nxt - loop.time()))
            await self.speaking(False)

In [0]:
fvc = VoiceClient(_FakeGC(), '1', AttrDict(id='2'))
fvc.sent,fvc.spoke = [],[]
fvc.send_pkt,fvc.encode = fvc.sent.append,lambda fr: b'E'
fvc.dave_version,fvc.ssrc = 0,1
class _SpkWS:
    @staticmethod
    async def send(msg): fvc.spoke.append(msg['d']['speaking'])
fvc.ws = _SpkWS()

await fvc.send_frames([b'a', b'b', b'c'])
assert fvc.sent == [b'E']*3 + [opus_silence]*5 and fvc.spoke == [1, 0]

fvc.sent.clear(); fvc.spoke.clear()
await fvc.send_frames([b'a'], end=False)    # mid-stream chunk: stay speaking, no silence tail
assert fvc.sent == [b'E'] and fvc.spoke == [1]

Discord voice expects 20 ms Opus frames. Padding the final partial frame preserves the tail of short clips instead of silently dropping it.

In [ ]:
#| export
frame_bytes = spf * n_chs * 2
def pcm2frames(pcm):
    for i in range(0, len(pcm), frame_bytes):
        fr = pcm[i:i+frame_bytes]
        yield fr if len(fr) == frame_bytes else fr + silence((frame_bytes - len(fr)) // (n_chs * 2))

@patch
async def send_pcm(self:VoiceClient, pcm, end=True): await self.send_frames(pcm2frames(pcm), end=end)

In [ ]:
t = np.arange(sr) / sr
mono = (0.08 * np.sin(2*np.pi*440*t) * 32767).astype(np.int16)
pcm = np.column_stack([mono, mono]).ravel().tobytes()
len(pcm)

192000

In [ ]:
await vc.send_pcm(pcm)

In [ ]:
await vc._reconnect(resume=True)
await asyncio.wait_for(vc.resumed.wait(), 5)
await vc.send_pcm(pcm)

To play audio back from a file, we've created a little helper using `ffmpeg`:

In [ ]:
#| export
def file2pcm(path):
    return ffmpeg.input(path).output('pipe:', f='s16le', ar=sr, ac=n_chs).run(capture_stdout=True, quiet=True)[0]

@patch
async def play_file(self:VoiceClient, path): await self.send_pcm(file2pcm(path))

In [ ]:
await vc.play_file("/tmp/recording_mixed.mp3")

Recording can continue across reconnect types.

In [ ]:
vc.start_recording()
await asyncio.sleep(2)
procs_before, start_before = dict(vc._rec_procs), vc._rec_start
await vc._reconnect(resume=False)
await asyncio.sleep(3)
await vc.stop_recording()

In [ ]:
await vc.play_file("/tmp/recording_mixed.mp3")

To leave a voice channel we first send a main-gateway voice state update with `channel_id=None`, which tells Discord the bot is leaving the channel.

After that we cancel the voice background tasks, close the voice WebSocket and UDP transport, and reset audio state. The reset matters because SSRC mappings, Opus decoders, RTP timestamps, and DAVE transition state are all tied to the old voice session.

In [ ]:
#| export
@patch
async def disconnect(self:VoiceClient):
    "Tear down voice ws/UDP without notifying the main gateway (used when Discord already removed us)"
    self.running = False
    for t in (self._hb_task, self._listen_task, self._ka_task):
        if t: t.cancel()
    if self.ws: await self.ws.close()
    if getattr(self, 'trans', None): self.trans.close()
    self.reset_audio()

@patch
async def leave(self:VoiceClient):
    self.running = False
    await self.gc.ws.send(Op.voice_state(self.gid, None))  # channel_id null = leave
    await self.disconnect()

In [ ]:
await vc.leave()

In [ ]:
await gc.stop()

## Bots

`Bot` ties together `DiscordClient` (REST) and `GatewayClient` (events) into a single object with a decorator-based command router. Commands are registered with `@bot.cmd` — the function name becomes the command name, prefixed with `!` in Discord. So `def echo` responds to `!echo`. Every command handler takes two arguments: the `Message` object and a string of everything the user typed after the command name (empty string if nothing).

The `_on_msg` handler ignores messages from the bot itself to prevent infinite loops — a common gotcha with Discord bots. It splits the message into command name + args, so `!echo hello world` passes `"hello world"` as the `args` string to the handler.

In [ ]:
#| export
class Bot:
    "Discord bot with command routing"
    def __init__(self, intents, **kw):
        self.dc = DiscordClient(**kw)
        self.gc = GatewayClient(intents, self.dc)
        self.cmds = {}
        self.errors = []
        self._on_err = None

    def cmd(self, f):
        "Register a command handler — function name becomes !name"
        self.cmds[f.__name__] = f
        return f

    async def _on_msg(self, msg):
        if msg.author['id'] == self.gc.user_id: return
        parts = msg.content.split(maxsplit=1)
        if not parts or not parts[0].startswith('!'): return
        name = parts[0][1:]
        if name not in self.cmds: return
        try:
            res = self.cmds[name](msg, parts[1] if len(parts) > 1 else '')
            if asyncio.iscoroutine(res): await res
        except Exception as e:
            self.errors.append(e)
            if self._on_err: await self._on_err(msg, e)

    async def start(self):
        "Connect to gateway and start listening"
        await self.gc.start()
        self.gc.on('MESSAGE_CREATE', self._on_msg)

    async def stop(self): await self.gc.stop()
    def __repr__(self): return f"Bot(cmds={list(self.cmds.keys())})"

In [ ]:
bot = Bot(intents)
await bot.start()

Connected! Session: aebd851f6f7e226c5f982f03cf7d27ef, heartbeat: 41250ms
Gateway started!


Commands can be registered at any time — even after `bot.start()`. This works because `@bot.cmd` just adds the function to a dict; the message handler looks up commands dynamically on each message.

In [ ]:
@bot.cmd
async def echo(msg, args): await (await msg.channel()).send(f'You said: {args}')

In [ ]:
bot

Bot(cmds=['echo'])

Errors in command handlers are caught and stored in `bot.errors` — useful for debugging in dynamic environments like solveit where you can inspect the list after the fact. For real-time handling (e.g. notifying the user in Discord), register a handler with `@bot.on_error`. Both mechanisms work simultaneously.

In [ ]:
#| export
@patch
def on_error(self:Bot, f):
    self._on_err = f
    return f

In [ ]:
@bot.on_error
async def handle_err(msg, e): print('error')

In [ ]:
@bot.cmd
def err(msg): raise Exception('test')

In [ ]:
bot.errors

[]

Voice integration reuses the existing `VoiceClient` — `Bot` just provides convenience methods to manage the lifecycle. The bot can only be in one voice channel at a time.

In [ ]:
#| export
@patch
async def join_voice(self:Bot, channel):
    "Join a voice channel and return VoiceClient"
    self.vc = VoiceClient(self.gc, channel)
    await self.vc.join()
    return self.vc

@patch
async def leave_voice(self:Bot):
    "Leave the current voice channel"
    if self.vc: await self.vc.leave()
    self.vc = None

In [ ]:
vc = await bot.join_voice(vch); vc

Voice ready!


VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2), self.running=True)

In [ ]:
vc.start_recording()

'recording.mp3'

In [ ]:
pth = vc.stop_recording()
await bot.leave_voice()

In [ ]:
# Audio(pth)

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()